# 第 49 课：流式输出与背压
## SSE · 增量解码 · 客户端断开 · 有界缓冲

**目标：**把第 48 课“等完整结果”的接口，推进到能逐段显示、能判断是否收全、能处理慢消费者的本机服务。

本讲义包含全部实现，不导入配套 `.py`，不依赖前课文件。配套脚本也能独立运行。默认只使用 **Python 标准库**，实际启动回环 HTTP；不会下载模型。

默认后端输出明确标注的演示文本，**不是语言模型回答、token 吞吐量或 GPU 性能成绩**。可选 Qwen 路径默认关闭。制作环境未安装 Transformers，没有执行真实权重与 CUDA 流式推理。

**重要区分：生成结束 ≠ 服务端写出 ≠ 客户端收全 ≠ 答案正确。**

阅读顺序：先做协议实验，再读有界缓冲与状态机，最后运行 HTTP 和独立练习。

## 1．从轮询到事件流：到底改变了什么？

第 48 课在生成结束后取结果。本课由客户端先创建请求，再订阅一条持续返回数据的连接。

```text
POST /v1/requests → 请求 ID
GET  /v1/requests/{id}/events → ready → delta… → done
```

SSE 的响应类型是 `text/event-stream`，文本采用 UTF-8；事件块由空行分隔。它是**应用层的数据格式**，不是“每个 TCP 包就是一个事件”。一个网络读取可能只有半个事件，也可能包含多个事件。[HTML SSE 规范](https://html.spec.whatwg.org/multipage/server-sent-events.html)

本课采用一次订阅、一次生成的窄协议，没有自动重连或历史重放。与第 48 课相比，暂时只保留一个生成执行槽，不实现等待推理队列：首次订阅后启动生成，等待订阅也受请求截止时间约束。

这不是上一课的原地兼容升级，而是一个可独立研究流式机制的小服务。

In [1]:
from __future__ import annotations

# %% 01 协议与工具函数
import argparse
import codecs
import hashlib
import hmac
import http.client
import json
import math
import os
import platform
import queue
import re
import secrets
import socket
import sys
import threading
import time
import uuid
from collections import deque
from contextlib import contextmanager
from dataclasses import dataclass
from datetime import datetime, timezone
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from typing import Any, Callable
from urllib.parse import urlsplit

FORMAT = 'lesson49-stream-v1'
MODEL_ID = 'Qwen/Qwen3-0.6B'
REVISION = 'c1899de289a04d12100db370d81485cdf75e47ca'
TERMINAL = frozenset({'succeeded', 'cancelled', 'expired', 'failed'})
SOURCES = {
    'sse': 'https://html.spec.whatwg.org/multipage/server-sent-events.html',
    'codecs': 'https://docs.python.org/3.13/library/codecs.html',
    'queue': 'https://docs.python.org/3.13/library/queue.html',
    'socket': 'https://docs.python.org/3.13/library/socket.html',
    'http': 'https://docs.python.org/3.13/library/http.server.html',
    'streamer': 'https://huggingface.co/docs/transformers/v4.57.3/en/internal/generation_utils',
    'streamer_source': 'https://github.com/huggingface/transformers/blob/v4.57.3/src/transformers/generation/streamers.py',
    'qwen': f'https://huggingface.co/{MODEL_ID}/blob/{REVISION}/README.md',
}


class ProtocolError(ValueError):
    pass


class ServiceError(ValueError):
    def __init__(self, status: int, code: str):
        super().__init__(code)
        self.status, self.code = status, code


class StreamStopped(RuntimeError):
    pass


class SlowConsumer(StreamStopped):
    pass


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ProtocolError(message)


def canonical(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False)


def strict_json(text: str) -> Any:
    def pairs(items):
        out = {}
        for key, value in items:
            require(key not in out, 'duplicate JSON key')
            out[key] = value
        return out
    def bad(_):
        raise ProtocolError('nonfinite JSON number')
    def number(s):
        value = float(s)
        if not math.isfinite(value):
            bad(s)
        return value
    try:
        return json.loads(text, object_pairs_hook=pairs,
                          parse_constant=bad, parse_float=number)
    except (json.JSONDecodeError, RecursionError, OverflowError) as exc:
        raise ProtocolError('invalid JSON') from exc


def text_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def split_utf8(text: str, max_bytes: int = 256) -> list[str]:
    """按Unicode码点切分，保证每段合法UTF-8；不保证完整词或字素簇。"""
    require(type(text) is str, 'text must be str')
    require(type(max_bytes) is int and max_bytes >= 4, 'max_bytes >= 4')
    result, current, size = [], [], 0
    for char in text:
        n = len(char.encode('utf-8', errors='strict'))
        if size + n > max_bytes:
            result.append(''.join(current))
            current, size = [], 0
        current.append(char)
        size += n
    if current:
        result.append(''.join(current))
    return result


def encode_event(sequence: int, event: str, payload: dict) -> bytes:
    """一条事件并非一个TCP包。JSON转义正文中的换行，避免注入SSE字段。"""
    require(type(sequence) is int and sequence > 0, 'positive sequence required')
    require(event in {'ready', 'delta', 'done'}, 'unknown event name')
    require(type(payload) is dict, 'object payload required')
    wire = f'id: {sequence}\nevent: {event}\ndata: {canonical(payload)}\n\n'
    return wire.encode('utf-8')

## 2．先看一条真实 SSE 事件

下面 `id` 是本次流的递增序号，`event` 指定类型，`data` 存放一行 JSON。末尾必须有空行。

我们将正文放进 JSON，正文中的换行由 JSON 编码器转义，不能通过回答内容偷偷插入新的 `event:` 字段。

`ready/delta/done` 是本课应用约定，不是所有 SSE 服务共用的内置事件。`[DONE]` 也不是 SSE 标准中的特殊语法。

In [2]:
example_frame = encode_event(2, 'delta', {
    'request_id': 'example',
    'text': '你好🌍\n第二行',
})
print(example_frame.decode('utf-8'))
print('UTF-8 字节数：', len(example_frame))
print('正文含换行，但没有注入第二条 SSE 事件。')

id: 2
event: delta
data: {"request_id":"example","text":"你好🌍\n第二行"}


UTF-8 字节数： 82
正文含换行，但没有注入第二条 SSE 事件。


## 3．三种边界不能混淆

| 边界 | 表示什么 |
|---|---|
| 模型 token | 模型一次选择的编号，不保证对应完整字符或词 |
| 可显示的文本增量 | tokenizer/streamer 整理出的字符串，可包含多个 token |
| 网络读取片段 | TCP/HTTP 客户端这一轮读到的字节，不保证完整 UTF-8 或事件 |

例如，“中”的 UTF-8 是 `[228,184,173]`。逐字节分别 `decode(errors='replace')` 会损失信息；增量解码器可以保留不完整序列，等后续字节到齐再输出。

本课解析器先做增量 UTF-8 解码，再按行处理 SSE，直到空行才派发事件。它支持 LF、CR、CRLF、多行 `data`、注释和 BOM，但遇到非法 UTF-8 会严格报错；这比浏览器规范的替换解码更严格，**不是完整浏览器 EventSource 实现**。[Python codecs](https://docs.python.org/3.13/library/codecs.html)

In [3]:
raw = '中'.encode('utf-8')
decoder = codecs.getincrementaldecoder('utf-8')()
print('原始字节：', list(raw))
for byte in raw:
    print('新到一个字节：', byte, '→', repr(decoder.decode(bytes([byte]), final=False)))
print('最后刷新：', repr(decoder.decode(b'', final=True)))
print('错误地分别解码：', ''.join(bytes([b]).decode('utf-8', 'replace') for b in raw))

原始字节： [228, 184, 173]
新到一个字节： 228 → ''
新到一个字节： 184 → ''
新到一个字节： 173 → '中'
最后刷新： ''
错误地分别解码： ���


### 解析器与客户端收集器

`SSEParser` 负责字节和协议；`ReplyCollector` 负责本课的请求身份、事件序号、字节数、终态和文本摘要。

两者各有预算，避免无限长的单行、事件或累计正文。摘要用于发现漏段和意外损坏，不是签名、来源认证或语义正确性证明。

In [4]:
# %% 02 增量SSE解析：跨网络片段保留UTF-8与事件状态
class SSEParser:
    """支持LF/CR/CRLF、BOM、注释、多行data、id与retry字段。
    与浏览器不同：本课程遇到非法UTF-8就报错，不用替换字符继续。
    只负责语法，不自动重连；设置大小上限，不声称完整EventSource实现。
    """
    def __init__(self, max_event_bytes: int = 8192):
        require(type(max_event_bytes) is int and max_event_bytes > 0, 'bad parser limit')
        self.decoder = codecs.getincrementaldecoder('utf-8-sig')(errors='strict')
        self.limit = max_event_bytes
        self.line = ''
        self.data: list[str] = []
        self.event = ''
        self.last_id = ''
        self.retry_ms: int | None = None
        self.after_cr = False
        self.block_bytes = 0
        self.closed = False
        self.discarded_partial = False

    def _line(self) -> dict | None:
        line, self.line = self.line, ''
        if line == '':
            out = None
            if self.data:  # data: 空值仍然应产生事件
                out = {'event': self.event or 'message',
                       'id': self.last_id, 'data': '\n'.join(self.data)}
            self.data, self.event, self.block_bytes = [], '', 0
            return out
        if line.startswith(':'):
            return None
        name, sep, value = line.partition(':')
        if sep and value.startswith(' '):
            value = value[1:]
        if not sep:
            value = ''
        if name == 'data':
            self.data.append(value)
        elif name == 'event':
            self.event = value
        elif name == 'id' and '\x00' not in value:
            self.last_id = value
        elif name == 'retry' and re.fullmatch(r'[0-9]{1,12}', value):
            # 本课不重连；有界地保存重连提示，不据此重发推理。
            self.retry_ms = int(value)
        return None

    def feed(self, raw: bytes) -> list[dict]:
        require(not self.closed and type(raw) is bytes, 'parser closed or nonbytes input')
        events = []
        for char in self.decoder.decode(raw, final=False):
            if self.after_cr:
                self.after_cr = False
                if char == '\n':
                    continue
            self.block_bytes += len(char.encode('utf-8'))
            require(self.block_bytes <= self.limit, 'event/line exceeds parser budget')
            if char in '\r\n':
                out = self._line()
                if out is not None:
                    events.append(out)
                self.after_cr = char == '\r'
            else:
                self.line += char
        return events

    def finish(self) -> None:
        require(not self.closed, 'parser already closed')
        self.closed = True
        # 未完整的UTF-8是错误；未以空行结束的SSE块不应在EOF时强行派发。
        self.decoder.decode(b'', final=True)
        self.discarded_partial = bool(self.line or self.data or self.event)
        self.line, self.data, self.event = '', [], ''


class ReplyCollector:
    """客户端：必须接到合法done并核对计数与摘要，才称为收全。
    hash只检查意外损坏/漏段，不是签名、服务身份或内容正确性证明。
    """
    def __init__(self, request_id: str, max_text_bytes: int = 65536):
        self.request_id = request_id
        self.max_bytes = max_text_bytes
        self.next_sequence = 1
        self.ready = False
        self.parts: list[str] = []
        self.byte_count = 0
        self.delta_count = 0
        self.done: dict | None = None
        self.first_delta_at: float | None = None

    def accept(self, event: dict) -> None:
        require(self.done is None, 'event after done')
        require(event.get('id') == str(self.next_sequence), 'missing/duplicate/out-of-order event')
        obj = strict_json(event['data'])
        require(type(obj) is dict and obj.get('request_id') == self.request_id,
                'wrong request identity')
        kind = event['event']
        if kind == 'ready':
            require(not self.ready and self.next_sequence == 1, 'ready must be first')
            self.ready = True
        elif kind == 'delta':
            require(self.ready, 'delta before ready')
            text = obj.get('text')
            require(type(text) is str and bool(text), 'nonempty delta required')
            self.byte_count += len(text.encode('utf-8'))
            require(self.byte_count <= self.max_bytes, 'client text budget exceeded')
            self.parts.append(text)
            self.delta_count += 1
            if self.first_delta_at is None:
                self.first_delta_at = time.monotonic()
        elif kind == 'done':
            require(self.ready, 'done before ready')
            require(obj.get('generation_state') in TERMINAL, 'invalid terminal state')
            require(type(obj.get('delta_count')) is int and obj['delta_count'] == self.delta_count,
                    'delta count mismatch')
            require(type(obj.get('text_bytes')) is int and obj['text_bytes'] == self.byte_count,
                    'byte count mismatch')
            require(obj.get('sha256') == text_hash(''.join(self.parts)), 'text hash mismatch')
            require(type(obj.get('finish_reason')) is str, 'missing finish reason')
            self.done = obj
        else:
            raise ProtocolError('unsupported application event')
        self.next_sequence += 1

    def report(self) -> dict:
        return {'request_id': self.request_id, 'text': ''.join(self.parts),
                'delta_count': self.delta_count, 'text_bytes': self.byte_count,
                'transport_complete': self.done is not None,
                'generation_state': self.done['generation_state'] if self.done else 'unknown',
                'finish_reason': self.done['finish_reason'] if self.done else 'eof_without_done',
                'task_success': None}

In [5]:
parser = SSEParser()
events = []
for byte in example_frame:
    events.extend(parser.feed(bytes([byte])))
parser.finish()
print('网络 feed 次数：', len(example_frame))
print('完整事件数量：', len(events))
print(events)
assert strict_json(events[0]['data'])['text'] == '你好🌍\n第二行'
# assert 用于课堂观察；完整脚本的正式检查使用显式异常，不依赖断言开关。

网络 feed 次数： 82
完整事件数量： 1
[{'event': 'delta', 'id': '2', 'data': '{"request_id":"example","text":"你好🌍\\n第二行"}'}]


### 心跳不等于正文；EOF 不等于完成

以冒号开头的行是 SSE 注释，可以用于心跳，本课不把它计入文本和首段延迟。多行 `data:` 用换行拼接。

规范要求：如果流在事件中间结束，没有终止空行，不能把剩余半个事件强行派发。本课客户端进一步要求有效 `done`，才确认收到了完整的本次协议结果。[HTML SSE 解释规则](https://html.spec.whatwg.org/multipage/server-sent-events.html#event-stream-interpretation)

即使 HTTP 正常返回 200，并最终读到了 EOF，没有 `done` 仍是未收全。

In [6]:
p = SSEParser()
print('心跳产生的事件：', p.feed(b': ping\r\n\r\n'))
print('多行数据：', p.feed(b'data: first\r\ndata: second\r\n\r\n'))
p = SSEParser()
print('未结束的事件：', p.feed(b'event: done\ndata: {}\n'))
p.finish()
print('EOF 丢弃半个事件：', p.discarded_partial)
print('空收集器的结果：', ReplyCollector('example').report())

心跳产生的事件： []
多行数据： [{'event': 'message', 'id': '', 'data': 'first\nsecond'}]
未结束的事件： []
EOF 丢弃半个事件： True
空收集器的结果： {'request_id': 'example', 'text': '', 'delta_count': 0, 'text_bytes': 0, 'transport_complete': False, 'generation_state': 'unknown', 'finish_reason': 'eof_without_done', 'task_success': None}


## 4．流式生成为什么需要背压？

假设生产者每秒提供 200 个文本片段，消费者每秒只能处理 20 个。在没有限制的假设下，10 秒可以积压约 `(200-20)×10=1800` 个片段。

**只限制事件条数还不够**：一条消息可能比另一条大上千倍。本课同时限制编码后的字节数、事件数量和单事件大小。

Python `Queue(maxsize)` 限制的是项目数量；满队列时 `put()` 可以等待或超时。本课采用 `Condition + deque`，把字节预算也纳入同一个加锁判断。[queue 官方文档](https://docs.python.org/3.13/library/queue.html)

策略是：**短暂等待 → 超过生产者等待预算 → 协作取消整个请求**。不会丢弃任意文本后继续宣称正常完成。

In [7]:
# %% 03 有界事件缓冲：数量与编码后的字节数都受限
class BoundedEventBuffer:
    """队列之外保留一个有界终态槽；满队列也能登记终止，不用挤占/删除delta。
    get()弹出后还有一个writer持有的frame；这不是整个进程内存上限。
    """
    def __init__(self, max_items: int = 8, max_bytes: int = 4096,
                 max_frame_bytes: int = 1024):
        for n in (max_items, max_bytes, max_frame_bytes):
            require(type(n) is int and n > 0, 'positive buffer limits required')
        require(max_frame_bytes <= max_bytes, 'frame larger than buffer')
        self.max_items, self.max_bytes, self.max_frame_bytes = max_items, max_bytes, max_frame_bytes
        self.cv = threading.Condition()
        self.items: deque[bytes] = deque()
        self.bytes = self.peak_bytes = self.peak_items = self.wait_count = 0
        self.closed = False
        self.tail: bytes | None = None

    def put(self, frame: bytes, timeout: float, stop: threading.Event) -> None:
        require(type(frame) is bytes and 0 < len(frame) <= self.max_frame_bytes,
                'frame exceeds limit')
        require(math.isfinite(timeout) and timeout >= 0, 'bad put timeout')
        deadline = time.monotonic() + timeout
        waited = False
        with self.cv:
            while True:
                if stop.is_set() or self.closed:
                    raise StreamStopped('buffer closed/cancelled')
                if len(self.items) < self.max_items and self.bytes + len(frame) <= self.max_bytes:
                    self.items.append(frame)
                    self.bytes += len(frame)
                    self.peak_bytes = max(self.peak_bytes, self.bytes)
                    self.peak_items = max(self.peak_items, len(self.items))
                    self.cv.notify_all()
                    return
                if not waited:
                    self.wait_count += 1
                    waited = True
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    raise SlowConsumer('bounded wait exhausted')
                self.cv.wait(min(remaining, 0.02))

    def get(self, timeout: float = 0.1) -> bytes | None:
        deadline = time.monotonic() + timeout
        with self.cv:
            while True:
                if self.items:
                    frame = self.items.popleft()
                    self.bytes -= len(frame)
                    self.cv.notify_all()
                    return frame
                if self.closed:
                    tail, self.tail = self.tail, None
                    return tail
                left = deadline - time.monotonic()
                if left <= 0:
                    raise queue.Empty
                self.cv.wait(left)

    def finish(self, tail: bytes) -> None:
        require(type(tail) is bytes and 0 < len(tail) <= self.max_frame_bytes, 'bad terminal frame')
        with self.cv:
            require(not self.closed, 'buffer already finished')
            self.tail, self.closed = tail, True
            self.cv.notify_all()

    def wake(self) -> None:
        with self.cv:
            self.cv.notify_all()

    def snapshot(self) -> dict:
        with self.cv:
            return {'queued_frames': len(self.items), 'queued_bytes': self.bytes,
                    'peak_frames': self.peak_items, 'peak_bytes': self.peak_bytes,
                    'put_waits': self.wait_count, 'terminal_slot_bytes': len(self.tail or b''),
                    'max_frames': self.max_items, 'max_bytes': self.max_bytes,
                    'max_frame_bytes': self.max_frame_bytes}

In [8]:
tiny = BoundedEventBuffer(max_items=3, max_bytes=4, max_frame_bytes=4)
not_stopped = threading.Event()
tiny.put('中'.encode(), 0, not_stopped)
tiny.put(b'a', 0, not_stopped)
print(tiny.snapshot())
try:
    tiny.put(b'b', 0, not_stopped)
except SlowConsumer:
    print('字节预算已满：不能继续增加。')
print('按 FIFO 取回：', tiny.get(), tiny.get())

{'queued_frames': 2, 'queued_bytes': 4, 'peak_frames': 2, 'peak_bytes': 4, 'put_waits': 0, 'terminal_slot_bytes': 0, 'max_frames': 3, 'max_bytes': 4, 'max_frame_bytes': 4}
字节预算已满：不能继续增加。
按 FIFO 取回： b'\xe4\xb8\xad' b'a'


### 满队列时，怎样仍然记下终态？

如果把“结束标记”也硬塞进已经满了的队列，控制器可能一直等不到终止。反过来，挤掉一段正文来腾空间，会造成静默漏字。

本课为终态保留**一个独立、有大小上限的槽位**。已接纳的文本仍按序输出，随后输出带有实际状态的 `done`；消费者断连时，终态至少仍保存在服务状态中，不保证还能传出去。

```text
正常队列：最多 N 条、最多 B 字节
额外终态槽：最多一个 frame
```

这不是进程总内存上限。还有生产者当前片段、writer 已取出的片段、socket 缓冲、客户端文本、模型 KV cache、streamer 缓存及 Python 对象开销。

In [9]:
tiny = BoundedEventBuffer(max_items=1, max_bytes=8, max_frame_bytes=8)
tiny.put(b'prefix', 0, threading.Event())
tiny.finish(b'cancel')
print('先保留正文：', tiny.get())
print('再取终态：', tiny.get())
print('最后才是EOF：', tiny.get())

先保留正文： b'prefix'
再取终态： b'cancel'
最后才是EOF： None


## 5．文本 streamer 与后端

默认后端输出 11 个预先定义的片段，不按问题进行智能回答；它们叫“演示单位”，不是模型 token。

可选真实路径覆盖 `TextStreamer.on_finalized_text()`，把可显示文本直接提交给有界缓冲。没有再套一个默认无界的 `TextIteratorStreamer` 队列，因此慢消费者的压力能传回生成线程。

这只管理这一层队列；模型 streamer 本身仍可能为了完整词而保留 token 缓存。`TextStreamer` 的段数不等于生成 token 数，源码也将该接口标为可能变化的接口。[固定版本实现](https://github.com/huggingface/transformers/blob/v4.57.3/src/transformers/generation/streamers.py)

本课固定最多 256 个新 token、实际上下文上限 1024。正文总字节数另有限制，不借助它推断显存。

In [10]:
# %% 04 演示后端与可选真实TextStreamer桥接
class DemoBackend:
    name = 'deterministic-demo-no-language-model'
    unit = 'demo_text_unit_not_model_token'
    pieces = ('演示：', '你好', '，', '世界', '🌍', '。\n', '这是', '逐步', '到达', '的文本', '。')

    def __init__(self, delay: float = 0.015):
        require(0 <= delay <= 1, 'bad demo delay')
        self.delay = delay

    def generate(self, request: dict, emit: Callable[[str], None], stop: threading.Event) -> dict:
        for piece in self.pieces[:request['max_new_tokens']]:
            if stop.wait(self.delay):
                raise StreamStopped('cooperative stop')
            emit(piece)
        return {'finish_reason': 'demo_end' if request['max_new_tokens'] >= len(self.pieces)
                else 'demo_limit', 'model_generated_tokens': None}


class QwenBackend:
    """未在制作环境实测的可选路径。
    覆盖TextStreamer.on_finalized_text，直接写有界缓冲；不额外创建无界iterator队列。
    streamer的完整词缓存、token缓存、GPU张量和socket缓冲不属于事件队列预算。
    """
    name = 'qwen3-0.6b-text-streamer'
    unit = 'model_token'

    def __init__(self, device='cpu', local_files_only=False):
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        if device not in {'cpu', 'cuda'}:
            raise ValueError('device must be cpu or cuda')
        if device == 'cuda' and not torch.cuda.is_available():
            raise RuntimeError('明确请求CUDA，但CUDA不可用；不会回退演示后端')
        self.torch, self.device = torch, device
        self.tokenizer = AutoTokenizer.from_pretrained(
            MODEL_ID, revision=REVISION, local_files_only=local_files_only,
            trust_remote_code=False)
        dtype = torch.float16 if device == 'cuda' else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, revision=REVISION, torch_dtype=dtype,
            local_files_only=local_files_only, trust_remote_code=False).to(device).eval()

    def generate(self, request, emit, stop):
        from transformers import TextStreamer, StoppingCriteria, StoppingCriteriaList
        torch, tok = self.torch, self.tokenizer
        class Sink(TextStreamer):
            def on_finalized_text(self, text, stream_end=False):
                if text:
                    emit(text)
        class Cancel(StoppingCriteria):
            def __call__(self, input_ids, scores, **kwargs):
                return torch.full((input_ids.shape[0],), stop.is_set(),
                                  dtype=torch.bool, device=input_ids.device)
        messages = [{'role': 'system', 'content': '请用简短中文回答。'},
                    {'role': 'user', 'content': request['prompt']}]
        text = tok.apply_chat_template(messages, tokenize=False,
                                       add_generation_prompt=True, enable_thinking=False)
        inputs = tok(text, add_special_tokens=False, return_tensors='pt')
        width = inputs['input_ids'].shape[1]
        require(width + request['max_new_tokens'] <= 1024, 'real context budget exceeded')
        inputs = {key: value.to(self.device) for key, value in inputs.items()}
        if stop.is_set():
            raise StreamStopped('stopped during input preparation')
        try:
            with torch.inference_mode():
                out = self.model.generate(
                    **inputs, max_new_tokens=request['max_new_tokens'], do_sample=False,
                    streamer=Sink(tok, skip_prompt=True, skip_special_tokens=True,
                                  clean_up_tokenization_spaces=False),
                    stopping_criteria=StoppingCriteriaList([Cancel()]),
                    use_cache=True)
            new = out[0, width:].tolist()
            eos = self.model.generation_config.eos_token_id
            eos = eos if isinstance(eos, list) else [eos]
            reason = 'eos' if new and new[-1] in eos else 'length'
            return {'finish_reason': reason, 'model_generated_tokens': len(new)}
        finally:
            # 不能在尚有CUDA工作时提前把模型执行槽交给下一个请求。
            if self.device == 'cuda':
                torch.cuda.synchronize()

## 6．请求、生成与传输状态

```text
提交 → awaiting_stream → 首次订阅 → running
                                     ↓
                       取消/期限/背压 → stopping
                                     ↓ 后端真正退出
                      succeeded / cancelled / expired / failed
```

`stopping` 期间，生成执行槽仍被占用，不能提前接入下一次模型计算。

与之分开，传输状态记录“写调用都返回了”或“连接写入失败”；客户端自己的收全结论不会由服务端猜测。发送成功不等于客户端应用已经读取，更不等于用户已经看见。

本课只允许一个订阅者，无 SSE 重放。重复提交相同幂等键只返回原 ID，不重新生成；第二次订阅或携带 `Last-Event-ID` 的订阅会明确返回 409。

In [11]:
# %% 05 请求状态：生成、传输、客户端接收是三套不同事实
@dataclass(frozen=True)
class Limits:
    buffer_items: int = 8
    buffer_bytes: int = 4096
    frame_bytes: int = 1024
    producer_wait_s: float = 0.25
    heartbeat_s: float = 0.10
    socket_timeout_s: float = 0.30
    max_text_bytes: int = 65536
    max_records: int = 32


def validate_request(body: Any) -> dict:
    require(type(body) is dict and set(body) == {'prompt', 'max_new_tokens', 'timeout_ms'},
            'exact prompt/max_new_tokens/timeout_ms fields required')
    require(type(body['prompt']) is str and bool(body['prompt'].strip()), 'empty prompt')
    require(len(body['prompt'].encode('utf-8')) <= 4096, 'prompt byte limit')
    require(type(body['max_new_tokens']) is int and 1 <= body['max_new_tokens'] <= 256,
            'max_new_tokens must be 1..256')
    require(type(body['timeout_ms']) is int and 50 <= body['timeout_ms'] <= 120000,
            'timeout_ms must be 50..120000')
    return dict(body)


class StreamJob:
    def __init__(self, owner: 'StreamService', request_id: str, request: dict):
        self.owner, self.request_id, self.request = owner, request_id, request
        self.lock = threading.RLock()
        lim = owner.limits
        self.buffer = BoundedEventBuffer(lim.buffer_items, lim.buffer_bytes, lim.frame_bytes)
        self.stop = threading.Event()
        self.finished = threading.Event()
        self.created = time.monotonic()
        self.deadline = self.created + request['timeout_ms'] / 1000
        self.state = 'awaiting_stream'
        self.claimed = False
        self.started = False
        self.stop_reason: str | None = None
        self.delivery = 'not_opened'
        self.finish_reason: str | None = None
        self.next_seq = 1
        self.text_bytes = self.delta_count = 0
        self.hasher = hashlib.sha256()
        self.started_at = self.first_delta_at = self.ended_at = None
        self.model_tokens = None
        self.writer_frames = 0
        self.worker: threading.Thread | None = None

    def request_stop(self, reason: str) -> bool:
        with self.lock:
            if self.state in TERMINAL:
                return False
            if self.stop_reason is None:
                self.stop_reason = reason
            self.stop.set()
            self.state = 'stopping'
            not_started = not self.started
        self.buffer.wake()
        if not_started:
            self._finish({'finish_reason': reason, 'model_generated_tokens': None})
        return True

    def emit(self, text: str) -> None:
        for piece in split_utf8(text, 256):
            raw = piece.encode('utf-8')
            with self.lock:
                require(self.text_bytes + len(raw) <= self.owner.limits.max_text_bytes,
                        'generated text byte limit exceeded')
                seq = self.next_seq
            self.buffer.put(encode_event(seq, 'delta',
                {'request_id': self.request_id, 'text': piece}),
                self.owner.limits.producer_wait_s, self.stop)
            with self.lock:
                self.next_seq += 1
                self.text_bytes += len(raw)
                self.delta_count += 1
                self.hasher.update(raw)
                if self.first_delta_at is None:
                    self.first_delta_at = time.monotonic()

    def _finish(self, result: dict) -> None:
        with self.lock:
            if self.state in TERMINAL:
                return
            reason = self.stop_reason or result.get('finish_reason', 'backend_error')
            if self.stop_reason == 'deadline':
                self.state = 'expired'
            elif self.stop_reason:
                self.state = 'cancelled'
            elif reason == 'backend_error':
                self.state = 'failed'
            else:
                self.state = 'succeeded'
            self.finish_reason = reason
            self.model_tokens = result.get('model_generated_tokens')
            self.ended_at = time.monotonic()
            terminal = {'request_id': self.request_id, 'generation_state': self.state,
                        'finish_reason': reason, 'text_bytes': self.text_bytes,
                        'delta_count': self.delta_count, 'sha256': self.hasher.hexdigest(),
                        'model_generated_tokens': self.model_tokens,
                        'task_success': None}
            self.buffer.finish(encode_event(self.next_seq, 'done', terminal))
        # 只有后端已经退出（或从未启动）才能释放执行槽。
        self.owner.release(self)
        self.finished.set()

    def run(self) -> None:
        result = {'finish_reason': 'backend_error', 'model_generated_tokens': None}
        try:
            # ready 已在订阅认领的临界区中写入，避免刚订阅就取消时出现 done 先于 ready。
            if self.stop.is_set():
                raise StreamStopped('cancelled before backend start')
            result = self.owner.backend.generate(self.request, self.emit, self.stop)
        except SlowConsumer:
            self.request_stop('slow_consumer')
        except StreamStopped:
            if not self.stop.is_set():
                self.request_stop('cooperative_stop')
        except Exception:
            # 本地服务不向客户端泄漏堆栈或文件路径。
            result = {'finish_reason': 'backend_error', 'model_generated_tokens': None}
        finally:
            self._finish(result)

    def snapshot(self) -> dict:
        with self.lock:
            def elapsed(end):
                return None if end is None or self.started_at is None else (end - self.started_at) * 1000
            out = {'request_id': self.request_id, 'generation_state': self.state,
                   'stream_claimed': self.claimed, 'delivery_state': self.delivery,
                   'stop_reason': self.stop_reason, 'finish_reason': self.finish_reason,
                   'delta_count': self.delta_count, 'text_bytes': self.text_bytes,
                   'sha256': self.hasher.hexdigest(), 'writer_frames': self.writer_frames,
                   'server_first_delta_ms': elapsed(self.first_delta_at),
                   'backend_elapsed_ms': elapsed(self.ended_at),
                   'model_generated_tokens': self.model_tokens,
                   'client_received_all': None, 'task_success': None}
        out['buffer'] = self.buffer.snapshot()
        return out


class StreamService:
    """单个生成执行槽，无推理排队；首次订阅后才启动，等待订阅也占期限。
    这是为突出流式逻辑的简化，不是第48课的接口兼容升级。
    每条请求只允许一次订阅；幂等重试不重新生成，不提供断线重放。
    """
    def __init__(self, backend=None, limits: Limits | None = None):
        self.backend = backend or DemoBackend()
        self.limits = limits or Limits()
        self.lock = threading.RLock()
        self.jobs: dict[str, StreamJob] = {}
        self.keys: dict[str, tuple[str, str]] = {}
        self.active: str | None = None
        self.closed = False
        self.api_key = secrets.token_urlsafe(24)
        self.epoch = uuid.uuid4().hex
        self.monitor_stop = threading.Event()
        self.monitor = threading.Thread(target=self._monitor, daemon=True)
        self.monitor.start()

    def submit(self, body: Any, key: str) -> tuple[StreamJob, bool]:
        request = validate_request(body)
        require(type(key) is str and re.fullmatch(r'[A-Za-z0-9_.-]{1,64}', key) is not None,
                'invalid Idempotency-Key')
        signature = text_hash(canonical(request))
        with self.lock:
            if key in self.keys:
                previous, rid = self.keys[key]
                if previous != signature:
                    raise ServiceError(409, 'idempotency_conflict')
                return self.jobs[rid], False
            if self.closed:
                raise ServiceError(503, 'service_stopping')
            if self.active is not None:
                raise ServiceError(503, 'generation_slot_busy')
            if len(self.jobs) >= self.limits.max_records:
                raise ServiceError(503, 'ledger_full')
            rid = uuid.uuid4().hex
            job = StreamJob(self, rid, request)
            self.jobs[rid] = job
            self.keys[key] = (signature, rid)
            self.active = rid
            return job, True

    def get(self, rid: str) -> StreamJob:
        with self.lock:
            if rid not in self.jobs:
                raise ServiceError(404, 'not_found')
            return self.jobs[rid]

    def claim(self, rid: str, last_event_id: str | None = None) -> StreamJob:
        job = self.get(rid)
        with job.lock:
            if last_event_id is not None:
                raise ServiceError(409, 'replay_not_supported')
            if job.claimed:
                raise ServiceError(409, 'stream_already_claimed_no_replay')
            if job.state != 'awaiting_stream' or time.monotonic() >= job.deadline:
                raise ServiceError(409, 'request_not_streamable')
            ready = {'request_id': job.request_id, 'backend': self.backend.name,
                     'unit': self.backend.unit, 'partial_until_done': True}
            job.buffer.put(encode_event(1, 'ready', ready), 0, job.stop)
            job.next_seq = 2
            job.claimed = job.started = True
            job.state, job.delivery = 'running', 'open'
            job.started_at = time.monotonic()
            job.worker = threading.Thread(target=job.run, daemon=True,
                                          name=f'generate-{rid[:8]}')
            job.worker.start()
        return job

    def release(self, job: StreamJob) -> None:
        with self.lock:
            if self.active == job.request_id:
                self.active = None

    def _monitor(self) -> None:
        while not self.monitor_stop.wait(0.01):
            with self.lock:
                jobs = list(self.jobs.values())
            for job in jobs:
                if time.monotonic() >= job.deadline:
                    job.request_stop('deadline')

    def close(self) -> None:
        with self.lock:
            self.closed = True
            jobs = list(self.jobs.values())
        for job in jobs:
            job.request_stop('service_shutdown')
        self.monitor_stop.set()
        self.monitor.join(timeout=1)
        # 这是有界等待，不是强杀线程。真实GPU挂起仍需外部进程级机制。
        for job in jobs:
            if job.worker:
                job.worker.join(timeout=2)

## 7．HTTP writer 与客户端

真实路径是 `POST` 创建、`GET` 订阅。示例使用 Python 客户端携带 Bearer 头，而不是浏览器原生 `EventSource`；后者的自动重连行为与本课的一次性契约不同。

SSE 是正文格式，本课用 HTTP/1.0 的连接关闭来界定流，不伪造 `Transfer-Encoding: chunked`。HTTP/1.1 分块传输、SSE 事件、TCP 读取，属于不同层。

`flush()` 只推动当前应用输出，不能保证代理和客户端完全不缓冲。写 socket 设置了超时，遇到超时/断连就向生成器发出停止信号。

连接半开或 GPU 内核挂起不能保证立即发现和终止；需要额外的进程/服务级隔离才能提供更强保证。[socket 文档](https://docs.python.org/3.13/library/socket.html)

In [12]:
# %% 06 HTTP适配：POST提交 + GET SSE，事件不是HTTP chunk
class StreamHTTPServer(ThreadingHTTPServer):
    daemon_threads = True
    allow_reuse_address = True
    request_queue_size = 8

    def __init__(self, address, service: StreamService):
        self.service = service
        super().__init__(address, StreamHandler)

    def get_request(self):
        conn, address = super().get_request()
        conn.settimeout(2.0)
        return conn, address


class StreamHandler(BaseHTTPRequestHandler):
    # 用HTTP/1.0连接关闭界定流响应；不伪造Transfer-Encoding:chunked。
    protocol_version = 'HTTP/1.0'
    server_version = 'Lesson49/1'
    wbufsize = 0

    def log_message(self, *_):
        pass

    def _auth(self):
        supplied = self.headers.get('Authorization', '')
        expected = 'Bearer ' + self.server.service.api_key
        if not hmac.compare_digest(supplied.encode(), expected.encode()):
            raise ServiceError(401, 'unauthorized')

    def _json(self, status, obj):
        raw = canonical(obj).encode('utf-8')
        self.send_response(status)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(raw)))
        self.send_header('Connection', 'close')
        self.end_headers()
        self.wfile.write(raw)
        self.close_connection = True

    def _body(self):
        if self.headers.get('Transfer-Encoding'):
            raise ServiceError(400, 'request_chunked_not_supported')
        lengths = self.headers.get_all('Content-Length', [])
        require(len(lengths) == 1 and re.fullmatch(r'[0-9]{1,6}', lengths[0]) is not None,
                'one Content-Length required')
        n = int(lengths[0])
        require(0 < n <= 8192, 'request body size limit')
        raw = self.rfile.read(n)
        require(len(raw) == n, 'truncated request body')
        return strict_json(raw.decode('utf-8', errors='strict'))

    def _handle(self, post=False):
        try:
            self._auth()
            require('?' not in self.path and '#' not in self.path, 'query parameters not supported')
            service = self.server.service
            if not post and self.path == '/healthz':
                self._json(200, {'service_epoch': service.epoch, 'backend': service.backend.name})
            elif post and self.path == '/v1/requests':
                job, created = service.submit(self._body(), self.headers.get('Idempotency-Key', ''))
                self._json(202 if created else 200, job.snapshot())
            else:
                m = re.fullmatch(r'/v1/requests/([0-9a-f]{32})(/events|/cancel)?', self.path)
                if not m:
                    raise ServiceError(404, 'unknown_route')
                rid, action = m.groups()
                if post and action == '/cancel':
                    require(self._body() == {}, 'cancel body must be {}')
                    job = service.get(rid)
                    job.request_stop('client_cancel')
                    self._json(200 if job.finished.is_set() else 202, job.snapshot())
                elif not post and action == '/events':
                    job = service.claim(rid, self.headers.get('Last-Event-ID'))
                    self._stream(job)
                elif not post and action is None:
                    self._json(200, service.get(rid).snapshot())
                else:
                    raise ServiceError(405, 'method_not_allowed')
        except ServiceError as exc:
            self._json(exc.status, {'error': exc.code})
        except (ProtocolError, UnicodeError):
            self._json(400, {'error': 'invalid_request'})
        except (OSError, TimeoutError):
            self.close_connection = True

    def _stream(self, job: StreamJob):
        service = self.server.service
        self.close_connection = True
        self.connection.settimeout(service.limits.socket_timeout_s)
        completed_write = False
        try:
            self.send_response(200)
            self.send_header('Content-Type', 'text/event-stream; charset=utf-8')
            self.send_header('Cache-Control', 'no-cache, no-transform')
            self.send_header('X-Accel-Buffering', 'no')  # 提示而不是代理行为保证
            self.send_header('Connection', 'close')
            self.end_headers()
            while True:
                try:
                    frame = job.buffer.get(service.limits.heartbeat_s)
                except queue.Empty:
                    # 注释心跳不作为delta，不计文本和首段时间。
                    self.wfile.write(b': ping\n\n')
                    self.wfile.flush()
                    continue
                if frame is None:
                    completed_write = True
                    break
                self.wfile.write(frame)
                self.wfile.flush()
                with job.lock:
                    job.writer_frames += 1
        except (OSError, TimeoutError):
            with job.lock:
                job.delivery = 'write_failed_or_disconnected'
            job.request_stop('transport_closed')
        finally:
            with job.lock:
                if completed_write:
                    job.delivery = 'all_write_calls_returned_not_client_ack'
                elif job.delivery == 'open':
                    job.delivery = 'handler_aborted'
            if not completed_write:
                job.request_stop('transport_closed')

    def do_GET(self):
        self._handle(False)

    def do_POST(self):
        self._handle(True)


@contextmanager
def local_server(backend=None, limits=None):
    service = StreamService(backend, limits)
    server = StreamHTTPServer(('127.0.0.1', 0), service)
    worker = threading.Thread(target=lambda: server.serve_forever(poll_interval=0.02), daemon=True)
    worker.start()
    try:
        yield service, f'http://127.0.0.1:{server.server_port}'
    finally:
        service.close()
        server.shutdown()
        server.server_close()
        worker.join(timeout=2)

### 为什么使用 `read1()`？

客户端不能要求“每次先读满一个大块再显示”，否则应用自身就可能把流式变成等待。这里用 `read1()` 接收可用片段，每次交给同一个解析器。

拿到 `ready` 不算首个正文；第一次合法、非空 `delta` 才记录客户端首段时间。直到 `done` 的计数与摘要通过，才记录 `transport_complete=True`。

即使终态是 `cancelled`，也可能成功收全了“部分文本＋取消结论”。这与成功生成整段答案不是同一件事。

In [13]:
# %% 07 标准库客户端：read1 + 增量解析 + 明确的done

def connection_for(url: str, timeout=5.0):
    parsed = urlsplit(url)
    require(parsed.scheme == 'http' and parsed.hostname in {'127.0.0.1', 'localhost'}
            and not parsed.username and not parsed.password and parsed.path in {'', '/'}
            and not parsed.query and not parsed.fragment, 'local HTTP base URL only')
    return http.client.HTTPConnection(parsed.hostname, parsed.port or 8765, timeout=timeout)


def json_call(url: str, key: str, method: str, path: str,
              body=None, idempotency_key=None, extra_headers=None) -> tuple[int, dict]:
    headers = {'Authorization': 'Bearer ' + key}
    if idempotency_key is not None:
        headers['Idempotency-Key'] = idempotency_key
    if extra_headers:
        headers.update(extra_headers)
    raw = None
    if body is not None:
        raw = canonical(body).encode('utf-8')
        headers['Content-Type'] = 'application/json'
    conn = connection_for(url)
    try:
        conn.request(method, path, body=raw, headers=headers)
        response = conn.getresponse()
        payload = response.read(131073)
        require(len(payload) <= 131072, 'JSON response too large')
        return response.status, strict_json(payload.decode('utf-8'))
    finally:
        conn.close()


def consume_stream(url: str, key: str, rid: str, *, read_size=17,
                   on_event=None, print_deltas=False, started_at=None) -> dict:
    require(type(read_size) is int and 1 <= read_size <= 4096, 'bad read size')
    start = started_at if started_at is not None else time.monotonic()
    parser, collector = SSEParser(), ReplyCollector(rid)
    conn = connection_for(url)
    response = None
    transport_error = None
    try:
        conn.request('GET', f'/v1/requests/{rid}/events', headers={
            'Authorization': 'Bearer ' + key, 'Accept': 'text/event-stream'})
        response = conn.getresponse()
        if response.status != 200:
            raise ProtocolError(f'SSE rejected: HTTP {response.status}: {response.read(4096)!r}')
        require(response.getheader('Content-Type', '').startswith('text/event-stream'), 'bad media type')
        while True:
            raw = response.read1(read_size)  # 不使用可能等待完整长度的read(大数)
            if not raw:
                parser.finish()
                break
            for event in parser.feed(raw):
                collector.accept(event)
                if print_deltas and event['event'] == 'delta':
                    print(strict_json(event['data'])['text'], end='', flush=True)
                if on_event:
                    on_event(event)
        # 没有done的正常HTTP EOF也不是完整结果。
    except (OSError, TimeoutError, http.client.HTTPException, ProtocolError, UnicodeError) as exc:
        transport_error = type(exc).__name__ + ': ' + str(exc)[:200]
    finally:
        if response is not None:
            response.close()
        conn.close()
    result = collector.report()
    if transport_error:
        result['transport_complete'] = False
    result.update({'transport_error': transport_error,
                   'discarded_partial_sse': parser.discarded_partial,
                   'client_first_delta_ms': None if collector.first_delta_at is None
                   else (collector.first_delta_at - start) * 1000,
                   'client_total_ms': (time.monotonic() - start) * 1000,
                   'metric_origin': 'client_before_submit_or_subscribe',
                   'automatic_reconnect': False})
    return result


def request_body(prompt='解释流式输出', max_new_tokens=64, timeout_ms=5000):
    return {'prompt': prompt, 'max_new_tokens': max_new_tokens, 'timeout_ms': timeout_ms}


def wait_for(predicate, timeout=3.0):
    deadline = time.monotonic() + timeout
    while not predicate():
        if time.monotonic() >= deadline:
            raise AssertionError('condition not reached before test deadline')
        time.sleep(0.003)

## 8．实验函数

下面包含六组可重复实验：Unicode/SSE、停滞消费者、取消等待、真实回环 HTTP、主动 TCP 断开、无读者 socket 发送超时。

HTTP 首段实验使用同步屏障，确保第一段到达客户端时后端仍未结束，不依赖偶然的线程调度。延时是人为设置，不用于推断模型性能。

In [14]:
# %% 08 可复现实验：不是模型能力或网络吞吐量基准

def unicode_demo() -> dict:
    raw = '中'.encode('utf-8')
    decoder = codecs.getincrementaldecoder('utf-8')()
    parts = [decoder.decode(bytes([b]), final=False) for b in raw]
    parts.append(decoder.decode(b'', final=True))
    wire = encode_event(1, 'delta', {'request_id': 'example', 'text': '你好🌍\n第二行'})
    parser = SSEParser()
    events = []
    for byte in wire:
        events.extend(parser.feed(bytes([byte])))
    parser.finish()
    return {'utf8_bytes': list(raw), 'incremental_parts': parts,
            'wrong_separate_decode': ''.join(bytes([b]).decode('utf-8', 'replace') for b in raw),
            'wire_bytes': len(wire), 'one_byte_feed_events': events,
            'network_read_count': len(wire), 'event_count': len(events)}


def drain_job(job: StreamJob) -> dict:
    parser, collector = SSEParser(), ReplyCollector(job.request_id)
    while True:
        frame = job.buffer.get(1.0)
        if frame is None:
            break
        for event in parser.feed(frame):
            collector.accept(event)
    parser.finish()
    return collector.report()


def backpressure_demo() -> dict:
    limits = Limits(buffer_items=2, buffer_bytes=512, producer_wait_s=0.04,
                    frame_bytes=512)
    service = StreamService(DemoBackend(0), limits)
    try:
        job, _ = service.submit(request_body(), 'no-reader')
        service.claim(job.request_id)
        # 有意不消费队列。模拟应用writer停滞，不伪称这是慢浏览器实测。
        require(job.finished.wait(2), 'backpressure did not finish')
        snapshot = job.snapshot()
        collected = drain_job(job)
        return {'kind': 'controlled_stalled_application_consumer',
                'snapshot_before_drain': snapshot, 'client_after_drain': collected,
                'no_silent_success': collected['generation_state'] == 'cancelled'}
    finally:
        service.close()


class GateBackend:
    """测试用：收到停止信号后还需外部释放，检验不能提前释放生成槽。"""
    name, unit = 'controlled-gate-test', 'not_tokens'
    def __init__(self):
        self.entered, self.observed_stop, self.release = (threading.Event() for _ in range(3))
    def generate(self, request, emit, stop):
        self.entered.set()
        stop.wait(2)
        self.observed_stop.set()
        if not self.release.wait(3):
            raise RuntimeError('test gate not released')
        return {'finish_reason': 'test_end', 'model_generated_tokens': None}


def cancellation_demo() -> dict:
    backend = GateBackend()
    service = StreamService(backend)
    try:
        job, _ = service.submit(request_body(), 'gate')
        service.claim(job.request_id)
        require(backend.entered.wait(1), 'gate not entered')
        job.request_stop('client_cancel')
        require(backend.observed_stop.wait(1), 'stop not seen')
        before = job.snapshot()
        active_before = service.active == job.request_id
        backend.release.set()
        require(job.finished.wait(2), 'not finished')
        return {'before_backend_exit': before, 'slot_still_held': active_before,
                'after_backend_exit': job.snapshot(), 'slot_released': service.active is None,
                'client': drain_job(job)}
    finally:
        backend.release.set()
        service.close()


def http_demo() -> dict:
    # 同步屏障保证首段被客户端看到时，生产者尚未完成；不依赖机器快慢。
    first_seen = threading.Event()
    class ProbeDemo(DemoBackend):
        def generate(self, request, emit, stop):
            first = True
            def tracked(text):
                nonlocal first
                emit(text)
                if first:
                    first = False
                    deadline = time.monotonic() + 2
                    while not first_seen.wait(0.01):
                        if stop.is_set() or time.monotonic() >= deadline:
                            raise StreamStopped('probe not observed')
            return super().generate(request, tracked, stop)
    with local_server(ProbeDemo(0.012)) as (service, url):
        start = time.monotonic()
        status, accepted = json_call(url, service.api_key, 'POST', '/v1/requests',
                                    request_body(), 'normal')
        require(status == 202, 'submit failed')
        rid = accepted['request_id']
        observed = []
        def observer(event):
            if event['event'] == 'delta' and not observed:
                observed.append(service.get(rid).snapshot()['generation_state'])
                first_seen.set()
        client = consume_stream(url, service.api_key, rid, read_size=7,
                                on_event=observer, started_at=start)
        wait_for(lambda: service.get(rid).snapshot()['delivery_state'] != 'open')
        _, snapshot = json_call(url, service.api_key, 'GET', f'/v1/requests/{rid}')
        replay_status, replay = json_call(url, service.api_key, 'GET',
                                          f'/v1/requests/{rid}/events',
                                          extra_headers={'Last-Event-ID': '2'})
        repeated_status, repeated = json_call(url, service.api_key, 'POST', '/v1/requests',
                                              request_body(), 'normal')
        return {'submit_status': status, 'first_delta_seen_while': observed,
                'client': client, 'server': snapshot,
                'resume_status': replay_status, 'resume_error': replay,
                'idempotent_status': repeated_status,
                'same_request_id': repeated['request_id'] == rid,
                'only_one_record': len(service.jobs) == 1,
                'backend': service.backend.name, 'measured_http': True,
                'first_delta_probe_uses_barrier': True}


class EndlessDemo:
    name, unit = 'disconnect-test-not-model', 'not_tokens'
    def generate(self, request, emit, stop):
        for _ in range(2000):
            if stop.wait(0.004):
                raise StreamStopped('disconnected')
            emit('测试断连' * 32)
        return {'finish_reason': 'test_end', 'model_generated_tokens': None}


def disconnect_demo() -> dict:
    with local_server(EndlessDemo()) as (service, url):
        _, obj = json_call(url, service.api_key, 'POST', '/v1/requests', request_body(), 'abort')
        rid = obj['request_id']
        conn = connection_for(url)
        conn.connect()
        sock = conn.sock
        conn.request('GET', f'/v1/requests/{rid}/events',
                     headers={'Authorization': 'Bearer ' + service.api_key})
        response = conn.getresponse()
        parser = SSEParser()
        received_delta = False
        while not received_delta:
            block = response.read1(64)
            require(bool(block), 'unexpected EOF in disconnect test')
            received_delta = any(e['event'] == 'delta' for e in parser.feed(block))
        # 主动关闭实际TCP方向；不声称可立即发现所有半开网络故障。
        sock.shutdown(socket.SHUT_RDWR)
        response.close()
        conn.close()
        job = service.get(rid)
        require(job.finished.wait(3), 'disconnect not detected in local test')
        return {'actual_loopback_disconnect': True, 'received_partial_delta': received_delta,
                'client_received_done': False, 'server': job.snapshot(),
                'slot_released': service.active is None}


def socket_timeout_demo() -> dict:
    """真实本机socket无读者实验，不等于生产网络/浏览器背压基准。"""
    left, right = socket.socketpair()
    try:
        left.setsockopt(socket.SOL_SOCKET, socket.SO_SNDBUF, 4096)
        left.settimeout(0.04)
        timed_out = False
        begin = time.monotonic()
        try:
            for _ in range(16):
                left.sendall(b'x' * 65536)
        except (TimeoutError, socket.timeout):
            timed_out = True
        return {'timed_out': timed_out, 'elapsed_ms': (time.monotonic()-begin)*1000,
                'meaning': 'send timeout; not proof of delivery or a model benchmark'}
    finally:
        left.close()
        right.close()

### 实验 A：真实 HTTP 流式接收

检查完整拼接、首段到达时生成仍在进行、客户端核对的摘要，以及本课明确不支持的重连。

只说明本次本机回环实验，不代表浏览器、反向代理、公网或负载均衡器表现。

In [15]:
http_result = http_demo()
print('完整文本：', repr(http_result['client']['text']))
print('delta 数：', http_result['client']['delta_count'])
print('UTF-8 字节数：', http_result['client']['text_bytes'])
print('第一段到达时：', http_result['first_delta_seen_while'])
print('收到并核对 done：', http_result['client']['transport_complete'])
print('服务端传输状态：', http_result['server']['delivery_state'])
print('重连请求状态码：', http_result['resume_status'])
print('客户端首段毫秒：', round(http_result['client']['client_first_delta_ms'], 3))
print('客户端总毫秒：', round(http_result['client']['client_total_ms'], 3))

完整文本： '演示：你好，世界🌍。\n这是逐步到达的文本。'
delta 数： 11
UTF-8 字节数： 62
第一段到达时： ['running']
收到并核对 done： True
服务端传输状态： all_write_calls_returned_not_client_ack
重连请求状态码： 409
客户端首段毫秒： 20.504
客户端总毫秒： 142.881


### 实验 B：应用写出端停滞

刻意不从应用缓冲取数据，使用 `2 条 / 512 字节` 的小容量和 `40ms` 有界等待。这里不是“慢浏览器”的测量，而是可控地验证应用层背压机制。

观察队列上限、取消原因和终态是否仍能读出。部分文本不能被标为完整成功答案。

In [16]:
slow_result = backpressure_demo()
print('缓冲快照：', slow_result['snapshot_before_drain']['buffer'])
print('生成状态：', slow_result['snapshot_before_drain']['generation_state'])
print('停止原因：', slow_result['snapshot_before_drain']['finish_reason'])
print('排空后客户端：', slow_result['client_after_drain'])

缓冲快照： {'queued_frames': 2, 'queued_bytes': 286, 'peak_frames': 2, 'peak_bytes': 286, 'put_waits': 1, 'terminal_slot_bytes': 295, 'max_frames': 2, 'max_bytes': 512, 'max_frame_bytes': 512}
生成状态： cancelled
停止原因： slow_consumer
排空后客户端： {'request_id': 'caa1b5bfdfa44f96ab4072f685e2d812', 'text': '演示：', 'delta_count': 1, 'text_bytes': 9, 'transport_complete': True, 'generation_state': 'cancelled', 'finish_reason': 'slow_consumer', 'task_success': None}


### 实验 C：取消信号不是后端完成

测试后端故意等候第二个释放信号。在它退出前，状态必须是 `stopping`，执行槽必须保持占用。随后才允许终态和下一份工作。

这不会强杀线程，也不会证明可强行中断 CUDA 内核。

In [17]:
cancel_result = cancellation_demo()
print('退出前：', cancel_result['before_backend_exit']['generation_state'])
print('槽位仍被占用：', cancel_result['slot_still_held'])
print('退出后：', cancel_result['after_backend_exit']['generation_state'])
print('槽位已释放：', cancel_result['slot_released'])

退出前： stopping
槽位仍被占用： True
退出后： cancelled
槽位已释放： True


### 实验 D：真正关闭 TCP 连接

本课客户端在读取一个完整 delta 后主动关闭双向 socket。服务端后续写入检测到失败，向生成器发出协作停止。

因为连接已经关闭，客户端不会收到最终 `done`。服务端却可以记录生成已经取消——这是两端不同的事实，不需要强行写成一致的成功状态。

In [18]:
disconnect_result = disconnect_demo()
print('客户端曾收到部分正文：', disconnect_result['received_partial_delta'])
print('客户端收到终态：', disconnect_result['client_received_done'])
print('服务端状态：', disconnect_result['server']['generation_state'])
print('服务端原因：', disconnect_result['server']['finish_reason'])
print('执行槽释放：', disconnect_result['slot_released'])
print('无读者 socket 发送实验：', socket_timeout_demo())

客户端曾收到部分正文： True
客户端收到终态： False
服务端状态： cancelled
服务端原因： transport_closed
执行槽释放： True
无读者 socket 发送实验： {'timed_out': True, 'elapsed_ms': 40.291718000048604, 'meaning': 'send timeout; not proof of delivery or a model benchmark'}


## 9．时间与完整性：不要制造“首 token”成绩

| 本课记录 | 真实含义 |
|---|---|
| `server_first_delta_ms` | 从生成线程启动，到第一段可显示文本进入应用队列 |
| `client_first_delta_ms` | 从客户端提交/订阅计时起点，到解析出第一段正文 |
| `client_total_ms` | 同一客户端计时起点，到流结束/失败 |
| `writer_frames` | 服务端写调用返回的帧数，不是客户端确认数 |

首文本片段可能合并多个 token，也可能等待完整词。上表不能直接称为模型原始 TTFT、tokens/s 或用户界面渲染延迟。

本课没有真实模型基准。延时、UUID、时间戳以及断连前已经写了几段都可能变化；验证的是顺序、状态、预算与正文完整性，不要求计时逐位相同。

## 10．重新连接，不等于断点续传

浏览器 EventSource 通常会在断开后重连，并可发送 `Last-Event-ID`。这只是游标信息，不会自动生成服务端日志、重放窗口或新的模型缓存。[规范的重连流程](https://html.spec.whatwg.org/multipage/server-sent-events.html#processing-model-8)

本课选择最小而诚实的契约：

- 一个请求只允许一个订阅者。
- 断连后协作取消，原生成不自动恢复。
- `Last-Event-ID` / 第二次订阅返回 409。
- 重复提交同一幂等键，只查原请求，不再生成一份相同答案。
- 服务重启后记录消失，没有跨重启幂等性。

未来要支持重放，应复用稳定请求与事件序号，保留完整事件日志，界定保留窗口、权限和消费游标；不能靠重新调用模型假装把剩余内容接上。

## 11．完整检查

测试涵盖 UTF-8、合并与拆分网络片段、SSE 空行、多行数据、EOF、事件序号、摘要、缓冲容量、取消、超时、实际本机 HTTP、主动断连以及重连拒绝。

检查使用显式异常，即使 Python 使用优化模式也不会悄悄移除。

In [19]:
# %% 09 固定检查：断言失败必须停止，测试不是模型能力成绩
class CheckBook:
    def __init__(self):
        self.names: list[str] = []
    def ok(self, name: str, condition: bool):
        if not condition:
            raise AssertionError(name)
        self.names.append(name)
    def raises(self, name: str, exception, fn: Callable[[], Any]):
        try:
            fn()
        except exception:
            self.names.append(name)
            return
        raise AssertionError(name + ': expected exception')


def check_suite() -> dict:
    c = CheckBook()
    # A. UTF-8与SSE分帧。每次feed的边界故意与字符/字段/事件不对齐。
    u = unicode_demo()
    c.ok('UTF8 three bytes', u['utf8_bytes'] == [228, 184, 173])
    c.ok('incremental decoder retains partial character', u['incremental_parts'] == ['', '', '中', ''])
    c.ok('separate byte decoding is lossy', u['wrong_separate_decode'] == '���')
    c.ok('one byte reads make one event', u['event_count'] == 1)
    c.ok('JSON preserves newline and emoji', strict_json(u['one_byte_feed_events'][0]['data'])['text'] == '你好🌍\n第二行')
    c.ok('network reads are not events', u['network_read_count'] > u['event_count'])
    for sep, title in [(b'\n', 'LF'), (b'\r', 'CR'), (b'\r\n', 'CRLF')]:
        raw = sep.join([b'id: 9', b'event: custom', b'data: first', b'data:second', b'', b''])
        p, result = SSEParser(), []
        for byte in raw:
            result.extend(p.feed(bytes([byte])))
        p.finish()
        c.ok('line endings '+title, result == [{'event':'custom','id':'9','data':'first\nsecond'}])
    p = SSEParser()
    c.ok('comments do not create events', p.feed(b': heartbeat\n\n') == [])
    c.ok('empty data field creates empty data event', p.feed(b'data:\n\n')[0]['data'] == '')
    c.ok('default event type', p.feed(b'data:hello\n\n')[0]['event'] == 'message')
    c.ok('strip only one space', p.feed(b'data:  x\n\n')[0]['data'] == ' x')
    c.ok('BOM consumed', SSEParser().feed(b'\xef\xbb\xbfdata: x\n\n')[0]['data'] == 'x')
    p = SSEParser()
    c.ok('multiple events in a read', len(p.feed(b'data:a\n\ndata:b\n\n')) == 2)
    p.feed(b'id: 4\n\n')
    c.ok('id persists across events', p.feed(b'data:x\n\n')[0]['id'] == '4')
    c.ok('NULL id ignored', p.feed(b'id: bad\x00id\ndata:y\n\n')[0]['id'] == '4')
    c.ok('empty id resets', p.feed(b'id:\ndata:z\n\n')[0]['id'] == '')
    p.feed(b'retry: 123\n\n')
    c.ok('retry parsed but not executed', p.retry_ms == 123)
    p.feed('retry: １２３\n\n'.encode())
    c.ok('retry must be ASCII digits', p.retry_ms == 123)
    p = SSEParser()
    c.ok('incomplete event not dispatched', p.feed(b'data: partial\n') == [])
    p.finish()
    c.ok('EOF discards pending event', p.discarded_partial)
    c.raises('invalid UTF8 is rejected', UnicodeError, lambda: SSEParser().feed(b'\xff'))
    p = SSEParser(); p.feed(b'\xe4')
    c.raises('partial UTF8 on EOF rejected', UnicodeError, p.finish)
    c.raises('bounded parser line', ProtocolError, lambda: SSEParser(8).feed(b'data: too long'))
    c.raises('unknown emitter event', ProtocolError, lambda: encode_event(1,'delta\nevent:done',{}))
    c.raises('emitter ID injection rejected', ProtocolError, lambda: encode_event('1\n', 'delta', {}))
    c.raises('boolean emitter ID rejected', ProtocolError, lambda: encode_event(True, 'delta', {}))
    c.raises('JSON duplicate key rejected', ProtocolError, lambda: strict_json('{"a":1,"a":2}'))
    c.raises('JSON NaN rejected', ProtocolError, lambda: strict_json('{"x":NaN}'))
    c.raises('JSON overflow rejected', ProtocolError, lambda: strict_json('{"x":1e999}'))
    text = '中a🌍e\u0301\n第二行'
    parts = split_utf8(text, 4)
    c.ok('UTF8 splitting exact reconstruction', ''.join(parts) == text)
    c.ok('UTF8 splitting byte limits', all(len(s.encode()) <= 4 for s in parts))
    # B. 应用级完整性：只有显式终态，不把EOF/200/heartbeat当成成功。
    def event(n, kind, data):
        return {'id': str(n), 'event': kind, 'data': canonical(data)}
    rid = 'test-request'
    ready = event(1,'ready',{'request_id':rid})
    delta = event(2,'delta',{'request_id':rid,'text':'中'})
    terminal = {'request_id':rid,'generation_state':'succeeded','finish_reason':'demo_end',
                'text_bytes':3,'delta_count':1,'sha256':text_hash('中')}
    done = event(3,'done',terminal)
    col = ReplyCollector(rid); col.accept(ready); col.accept(delta)
    c.ok('partial text is not completion', not col.report()['transport_complete'])
    col.accept(done)
    c.ok('valid done completes stream', col.report()['transport_complete'])
    c.ok('task success not inferred', col.report()['task_success'] is None)
    c.raises('no events after done', ProtocolError, lambda: col.accept(done))
    c.raises('delta before ready rejected', ProtocolError,
             lambda: ReplyCollector(rid).accept(event(1,'delta',{'request_id':rid,'text':'x'})))
    c.raises('wrong request rejected', ProtocolError, lambda: ReplyCollector('other').accept(ready))
    col = ReplyCollector(rid); col.accept(ready)
    c.raises('duplicate sequence rejected', ProtocolError, lambda: col.accept(ready))
    c.raises('gap rejected', ProtocolError, lambda: col.accept(event(3,'delta',{'request_id':rid,'text':'x'})))
    col = ReplyCollector(rid, 2); col.accept(ready)
    c.raises('client budget measures UTF8 bytes', ProtocolError, lambda: col.accept(delta))
    for field, value in [('sha256','0'*64), ('text_bytes',1), ('delta_count',2), ('generation_state','running')]:
        col = ReplyCollector(rid); col.accept(ready); col.accept(delta)
        bad = dict(terminal); bad[field] = value
        c.raises('terminal validates '+field, ProtocolError, lambda bad=bad,col=col: col.accept(event(3,'done',bad)))
    col = ReplyCollector(rid); col.accept(ready); col.accept(delta)
    stop_done = dict(terminal, generation_state='cancelled', finish_reason='slow_consumer')
    col.accept(event(3,'done',stop_done))
    c.ok('received cancellation is not successful generation',
         col.report()['transport_complete'] and col.report()['generation_state'] == 'cancelled')
    # C. 队列容量、阻塞唤醒、终态槽。
    stop = threading.Event()
    b = BoundedEventBuffer(3,4,4)
    b.put('中'.encode(),0,stop); b.put(b'a',0,stop)
    c.ok('buffer counts bytes not characters', b.snapshot()['queued_bytes'] == 4)
    c.raises('byte limit respected', SlowConsumer, lambda: b.put(b'b',0,stop))
    c.ok('FIFO retained', b.get() == '中'.encode() and b.get() == b'a')
    c.ok('pop releases byte reservation', b.snapshot()['queued_bytes'] == 0)
    c.raises('oversized single frame rejected', ProtocolError, lambda: b.put(b'12345',0,stop))
    b = BoundedEventBuffer(1,8,8); b.put(b'x',0,stop)
    c.raises('item limit independent of bytes', SlowConsumer, lambda: b.put(b'y',0,stop))
    b.finish(b'end')
    c.ok('terminal does not displace queued delta', b.get() == b'x')
    c.ok('terminal has reserved slot', b.get() == b'end' and b.get() is None)
    c.raises('put after finish rejected', StreamStopped, lambda: b.put(b'z',0,stop))
    c.raises('second finish rejected', ProtocolError, lambda: b.finish(b'again'))
    b = BoundedEventBuffer(1,8,8); b.put(b'x',0,stop)
    c.raises('empty consumer timeout', queue.Empty, lambda: BoundedEventBuffer().get(0))
    stopped = threading.Event(); errors = []
    def blocked_put():
        try: b.put(b'y',1,stopped)
        except StreamStopped as exc: errors.append(type(exc).__name__)
    worker = threading.Thread(target=blocked_put); worker.start()
    wait_for(lambda: b.snapshot()['put_waits'] == 1)
    stopped.set(); b.wake(); worker.join(1)
    c.ok('cancellation wakes blocked producer', not worker.is_alive() and errors == ['StreamStopped'])
    slow = backpressure_demo()
    c.ok('stalled writer stops producer', slow['snapshot_before_drain']['finish_reason'] == 'slow_consumer')
    c.ok('stalled writer cannot grow queue', slow['snapshot_before_drain']['buffer']['peak_frames'] <= 2
         and slow['snapshot_before_drain']['buffer']['peak_bytes'] <= 512)
    c.ok('terminal survives full buffer', slow['client_after_drain']['transport_complete'])
    c.ok('overflow never reported as success', slow['no_silent_success'])
    cancelled = cancellation_demo()
    c.ok('stopping until backend exits', cancelled['before_backend_exit']['generation_state'] == 'stopping')
    c.ok('generation slot not released early', cancelled['slot_still_held'])
    c.ok('slot released after exit', cancelled['slot_released'])
    c.ok('cancelled stream has consistent summary', cancelled['client']['generation_state'] == 'cancelled'
         and cancelled['client']['transport_complete'])
    # D. 请求、期限、失败和并发契约。
    c.raises('request boolean token limit rejected', ProtocolError,
             lambda: validate_request(request_body(max_new_tokens=True)))
    bad = request_body(); bad['approved'] = True
    c.raises('unexpected fields rejected', ProtocolError, lambda: validate_request(bad))
    c.raises('remote URL rejected', ProtocolError, lambda: connection_for('http://example.com'))
    service = StreamService(DemoBackend(0))
    try:
        job, created = service.submit(request_body(timeout_ms=70),'deadline')
        again, created2 = service.submit(request_body(timeout_ms=70),'deadline')
        c.ok('idempotent submit same record', created and not created2 and job is again)
        c.raises('same key different payload conflict', ServiceError,
                 lambda: service.submit(request_body(timeout_ms=100),'deadline'))
        c.raises('single generation admission', ServiceError,
                 lambda: service.submit(request_body(),'other'))
        c.raises('Last-Event-ID explicitly unsupported', ServiceError,
                 lambda: service.claim(job.request_id,'1'))
        c.ok('rejected replay does not claim stream', not job.claimed)
        c.ok('unsubscribed request expires', job.finished.wait(1) and job.state == 'expired')
        c.ok('unsubscribed request never started backend', not job.started)
        c.ok('unsubscribed expiry releases admission', service.active is None)
    finally:
        service.close()
    class Faulty:
        name, unit = 'fault-test','not_tokens'
        def generate(self, request, emit, stop):
            emit('部分')
            raise RuntimeError('do not publish internal path /secret')
    service = StreamService(Faulty())
    try:
        job,_=service.submit(request_body(),'fault'); service.claim(job.request_id)
        c.ok('backend exception finalized', job.finished.wait(1) and job.state == 'failed')
        report = drain_job(job)
        c.ok('backend failure retains prefix but not success', report['text']=='部分'
             and report['generation_state']=='failed' and report['transport_complete'])
        c.ok('failure sanitized', job.finish_reason == 'backend_error')
    finally: service.close()
    service = StreamService(DemoBackend(0.1))
    try:
        job,_=service.submit(request_body(timeout_ms=60),'exp');service.claim(job.request_id)
        c.ok('running deadline reaches terminal', job.finished.wait(1) and job.state=='expired')
        c.ok('expired generation retains no token fabrication', job.model_tokens is None)
    finally: service.close()
    service = StreamService(DemoBackend(0.1))
    try:
        job,_ = service.submit(request_body(),'immediate-stop')
        service.claim(job.request_id)
        job.request_stop('client_cancel')
        c.ok('immediate cancel reaches terminal', job.finished.wait(1) and job.state=='cancelled')
        c.ok('ready precedes done even on immediate cancel', drain_job(job)['transport_complete'])
    finally: service.close()
    # E. 真正的本机HTTP、断连与发送超时。
    net = http_demo()
    c.ok('HTTP accepted then collected', net['submit_status']==202 and net['client']['transport_complete'])
    c.ok('HTTP Chinese newline emoji exact', net['client']['text']==''.join(DemoBackend.pieces))
    c.ok('first delta arrives before terminal', net['first_delta_seen_while']==['running'])
    c.ok('application event count not tokens', net['client']['delta_count']==11 and net['server']['model_generated_tokens'] is None)
    c.ok('server does not claim client acknowledgement', net['server']['client_received_all'] is None)
    c.ok('server write status is not receiver proof', net['server']['delivery_state']=='all_write_calls_returned_not_client_ack')
    c.ok('network resume explicitly rejected', net['resume_status']==409)
    c.ok('repeated submit does not regenerate', net['same_request_id'] and net['only_one_record'])
    c.ok('local latency ordered', net['client']['client_total_ms'] >= net['client']['client_first_delta_ms'] > 0)
    disc=disconnect_demo()
    c.ok('actual TCP disconnect observed', disc['actual_loopback_disconnect'] and not disc['client_received_done'])
    c.ok('disconnect triggers cooperative stop', disc['server']['generation_state']=='cancelled'
         and disc['server']['finish_reason']=='transport_closed')
    c.ok('disconnect slot eventually released', disc['slot_released'])
    c.ok('local socket blocked send timeout tested', socket_timeout_demo()['timed_out'])
    with local_server() as (service,url):
        status,_=json_call(url,'wrong','GET','/healthz')
        c.ok('Bearer checked before endpoint handling', status==401)
        status,_=json_call(url,service.api_key,'GET','/missing')
        c.ok('unknown endpoint',status==404)
        status,_=json_call(url,service.api_key,'POST','/v1/requests',bad,'bad')
        c.ok('HTTP strict schema',status==400)
        _,obj=json_call(url,service.api_key,'POST','/v1/requests',request_body(),'cancel-wait')
        status,res=json_call(url,service.api_key,'POST',f"/v1/requests/{obj['request_id']}/cancel",{})
        c.ok('cancel before subscription terminates',status==200 and res['generation_state']=='cancelled')
    # 已经完成的生成不因之后的取消请求被改写。
    service=StreamService(DemoBackend(0), Limits(buffer_items=32,buffer_bytes=16384))
    try:
        job,_=service.submit(request_body(),'done');service.claim(job.request_id)
        require(job.finished.wait(1),'did not finish')
        c.ok('terminal cancellation does not rewrite outcome', not job.request_stop('client_cancel')
             and job.state=='succeeded')
        c.raises('second stream not silently restarted',ServiceError,lambda:service.claim(job.request_id))
    finally:service.close()
    return {'format':FORMAT, 'passed':len(c.names), 'names':c.names,
            'scope':'standard library + actual loopback HTTP; no model/GPU test'}

In [20]:
checks = check_suite()
print(f"固定检查通过：{checks['passed']} 项。")
print(checks['scope'])

固定检查通过：104 项。
standard library + actual loopback HTTP; no model/GPU test


## 12．保存实验报告

定义单元中的 `main()` 不会被自动调用。下面可以独立运行，不需要配套脚本文件。新的时间戳目录避免覆盖之前的实验。

报告保存实际测试环境与结果，不包含 API 密钥。对自己的提示运行时，注意日志中的明文资料。

In [21]:
# %% 10 实验报告与CLI；不把计时当作固定参考答案

def new_directory(root: str | Path) -> Path:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = Path(root) / ('local_' + stamp + '_' + secrets.token_hex(2))
    path.mkdir(parents=True, exist_ok=False)
    return path


def write_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')


def run_experiments(output_root='lesson_49_outputs', include_checks=True) -> Path:
    path = new_directory(output_root)
    manifest = {'format': FORMAT, 'python':sys.version, 'platform':platform.platform(),
                'backend':'standard-library demonstrations', 'actual_http':True,
                'real_qwen_tested':False, 'cuda_tested':False, 'status':'running',
                'sources':SOURCES}
    write_json(path/'manifest.json',manifest)
    try:
        if include_checks:
            checks=check_suite();write_json(path/'checks.json',checks)
        write_json(path/'protocol.json',unicode_demo())
        write_json(path/'backpressure.json',backpressure_demo())
        write_json(path/'cancellation.json',cancellation_demo())
        write_json(path/'http_stream.json',http_demo())
        write_json(path/'disconnect.json',disconnect_demo())
        write_json(path/'socket_timeout.json',socket_timeout_demo())
        manifest['status']='completed'
    except Exception as exc:
        manifest.update(status='failed',error=type(exc).__name__+': '+str(exc))
        raise
    finally:
        write_json(path/'manifest.json',manifest)
    return path


def doctor() -> dict:
    import importlib.util
    import importlib.metadata
    packages={}
    for name in ['torch','transformers','safetensors']:
        try: version=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: version=None
        packages[name]={'installed':bool(importlib.util.find_spec(name)), 'version':version}
    return {'python':sys.version,'platform':platform.platform(),'packages':packages,
            'default_requires':'Python standard library only',
            'real_model':MODEL_ID,'revision':REVISION,'real_loading_tested_here':False}


def serve_forever(backend, port: int):
    require(type(port) is int and 1 <= port <= 65535,'invalid port')
    service=StreamService(backend)
    server=StreamHTTPServer(('127.0.0.1',port),service)
    thread=threading.Thread(target=lambda:server.serve_forever(poll_interval=0.1),daemon=True)
    thread.start()
    print(canonical({'url':f'http://127.0.0.1:{port}', 'api_key':service.api_key,
                     'backend':backend.name, 'service_epoch':service.epoch,
                     'note':'local only; one subscriber; no replay; Ctrl+C to stop'}),flush=True)
    try:
        while thread.is_alive():
            time.sleep(0.2)
    except KeyboardInterrupt:
        print('\n请求停止；非硬取消。',flush=True)
    finally:
        service.close();server.shutdown();server.server_close();thread.join(timeout=2)


def main(argv=None):
    parser=argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--checks-only',action='store_true')
    parser.add_argument('--doctor',action='store_true')
    parser.add_argument('--output-dir',default='lesson_49_outputs')
    parser.add_argument('--serve',action='store_true')
    parser.add_argument('--port',type=int,default=8766)
    parser.add_argument('--backend',choices=['demo','qwen'],default='demo')
    parser.add_argument('--device',choices=['cpu','cuda'],default='cpu')
    parser.add_argument('--local-files-only',action='store_true')
    parser.add_argument('--client',help='http://127.0.0.1:8766')
    parser.add_argument('--key',default='request-001',help='幂等键，不是认证密钥')
    parser.add_argument('--prompt',default='用一句话解释KV cache。')
    parser.add_argument('--max-new-tokens',type=int,default=64)
    parser.add_argument('--timeout-ms',type=int,default=30000)
    parser.add_argument('--request-id')
    parser.add_argument('--cancel',action='store_true')
    parser.add_argument('--events',action='store_true',help='订阅尚未订阅的请求；不重放旧流')
    args=parser.parse_args(argv)
    if args.doctor:
        print(json.dumps(doctor(),ensure_ascii=False,indent=2));return
    if args.checks_only:
        result=check_suite()
        print(f"固定检查通过：{result['passed']}项。真实本地HTTP；不是Qwen/GPU或生产压测。")
        return
    if args.serve:
        backend=DemoBackend() if args.backend=='demo' else QwenBackend(args.device,args.local_files_only)
        serve_forever(backend,args.port);return
    if args.client:
        api_key=os.environ.get('LESSON49_API_KEY','')
        require(bool(api_key),'请设置 LESSON49_API_KEY，不要把凭证放入URL')
        if args.request_id:
            path=f'/v1/requests/{args.request_id}'
            if args.cancel:
                status,obj=json_call(args.client,api_key,'POST',path+'/cancel',{})
            elif args.events:
                obj=consume_stream(args.client,api_key,args.request_id,print_deltas=True)
                status=200 if obj['transport_complete'] else 409
                print()
            else:
                status,obj=json_call(args.client,api_key,'GET',path)
            print(json.dumps({'status':status,'result':obj},ensure_ascii=False,indent=2));return
        require(not args.cancel and not args.events,'cancel/events require --request-id')
        start=time.monotonic()
        status,obj=json_call(args.client,api_key,'POST','/v1/requests',
                            request_body(args.prompt,args.max_new_tokens,args.timeout_ms),args.key)
        print(json.dumps({'http_status':status,'request':obj},ensure_ascii=False),flush=True)
        if status not in {200,202}:
            raise ServiceError(status,obj.get('error','submission_failed'))
        if obj['stream_claimed'] or obj['generation_state']!='awaiting_stream':
            print('这是已有请求；本课不重放旧流，也不重新生成。用 --request-id 查询状态。');return
        result=consume_stream(args.client,api_key,obj['request_id'],print_deltas=True,started_at=start)
        print('\n'+json.dumps(result,ensure_ascii=False,indent=2));return
    path=run_experiments(args.output_dir)
    print('实验完成：'+str(path.resolve()))
    print('默认后端为演示文本，不是模型回答；断线重连、持久化和硬取消未实现。')

In [22]:
RUN_SAVE_REPORTS = True
if RUN_SAVE_REPORTS:
    saved_run = run_experiments('lesson_49_outputs', include_checks=False)
    write_json(saved_run / 'checks.json', checks)
    print('报告目录：', saved_run.resolve())
else:
    print('未写入报告。')

报告目录： /mnt/data/_lesson49_work/notebook_run/lesson_49_outputs/local_20261102T114639_359830Z_386e


## 13．独立练习：先写，再打开开关

**A．跨片段解析**：实现 `collect_fragments_ex(fragments)`。复用本课 `SSEParser`，但始终使用同一个实例，按顺序 feed，最后 finish。不能逐片直接解码，也不能把不完整 EOF 强行当作一条事件。

**B．双重容量判断**：实现 `can_enqueue_ex(items, used_bytes, frame_bytes, max_items, max_bytes)`。检查正整数与合法使用量，返回是否能同时满足数量和字节限制。等于容量可以，超过不行。这个纯函数可用于理解缓冲条件，不代替加锁与等待。

**C．终态核对**：实现 `verify_done_ex(parts, terminal, request_id)`。检查请求身份、真实终态、正文条数、UTF-8 字节数和摘要；返回传输完整性与生成状态。收全取消结果也不算成功生成，更不能推断任务正确。

正式客户端的事件序号检查仍由 `ReplyCollector` 负责。不要拿 C 的函数单独替代整个协议。

In [23]:
def collect_fragments_ex(fragments):
    raise NotImplementedError('练习A：保持同一个解析器')


def can_enqueue_ex(items, used_bytes, frame_bytes, max_items, max_bytes):
    raise NotImplementedError('练习B：同时检查两项容量')


def verify_done_ex(parts, terminal, request_id):
    raise NotImplementedError('练习C：核对真正接到的正文')

RUN_EXERCISES = False

### 练习检查器

检查器不调用参考答案生成预期；预期来自协议与手算数值。失败时会抛出异常。

In [24]:
def check_exercise_a(fn):
    wire = encode_event(1, 'delta', {'request_id':'r', 'text':'中🌍\n下一行'})
    expected = [{'id':'1','event':'delta','data':canonical({'request_id':'r','text':'中🌍\n下一行'})}]
    require(fn([bytes([b]) for b in wire]) == expected, 'A: byte fragmentation')
    require(fn([wire + wire]) == expected * 2, 'A: coalescing')
    require(fn([b'data: unfinished']) == [], 'A: unfinished event')
    require(fn([b':ping\n\n']) == [], 'A: comment')
    return 4


def check_exercise_b(fn):
    cases = [(0,0,3,2,4,True),(1,3,1,2,4,True),(1,3,2,2,4,False),
             (2,1,1,2,4,False),(0,0,5,2,4,False)]
    for a,b,c,d,e,expected in cases:
        require(fn(a,b,c,d,e) is expected, 'B: capacity boundary')
    try: fn(True,0,1,2,4)
    except (ValueError,TypeError): pass
    else: raise AssertionError('B: bool is not a count')
    try: fn(0,-1,1,2,4)
    except (ValueError,TypeError): pass
    else: raise AssertionError('B: negative usage')
    return 7


def check_exercise_c(fn):
    parts = ['中','a']
    terminal = {'request_id':'r','generation_state':'cancelled','finish_reason':'slow_consumer',
                'delta_count':2,'text_bytes':4,'sha256':text_hash('中a')}
    report = fn(parts,terminal,'r')
    require(report['transport_complete'] is True and report['generation_state']=='cancelled',
            'C: cancellation is a complete terminal, not successful generation')
    require(report['task_success'] is None, 'C: no fabricated task success')
    for field,value in [('request_id','other'),('delta_count',1),('text_bytes',2),
                        ('sha256','0'*64),('generation_state','running')]:
        bad=dict(terminal); bad[field]=value
        try:fn(parts,bad,'r')
        except (ValueError,TypeError):pass
        else:raise AssertionError('C failed to reject '+field)
    return 7

In [25]:
if RUN_EXERCISES:
    print('A检查：', check_exercise_a(collect_fragments_ex))
    print('B检查：', check_exercise_b(can_enqueue_ex))
    print('C检查：', check_exercise_c(verify_done_ex))
else:
    print('练习未执行；完整运行Notebook不等于完成了独立练习。')

练习未执行；完整运行Notebook不等于完成了独立练习。


## 14．参考答案

做到这里之前，先保存自己的实现。下面使用不同函数名，不覆盖你的练习代码。

In [26]:
def collect_fragments_ref(fragments):
    parser = SSEParser()
    result = []
    for raw in fragments:
        result.extend(parser.feed(raw))
    parser.finish()
    return result

print('参考A通过：', check_exercise_a(collect_fragments_ref))

参考A通过： 4


In [27]:
def can_enqueue_ref(items, used_bytes, frame_bytes, max_items, max_bytes):
    values = (items, used_bytes, frame_bytes, max_items, max_bytes)
    require(all(type(n) is int for n in values), 'counts must be integers, not bool')
    require(max_items > 0 and max_bytes > 0 and frame_bytes > 0, 'positive capacities/frame')
    require(0 <= items <= max_items and 0 <= used_bytes <= max_bytes, 'invalid existing usage')
    return items < max_items and used_bytes + frame_bytes <= max_bytes

print('参考B通过：', check_exercise_b(can_enqueue_ref))

参考B通过： 7


In [28]:
def verify_done_ref(parts, terminal, request_id):
    require(type(parts) is list and all(type(p) is str and p for p in parts), 'nonempty string parts')
    require(type(terminal) is dict and terminal.get('request_id') == request_id, 'wrong request')
    require(terminal.get('generation_state') in TERMINAL, 'not terminal')
    text = ''.join(parts)
    require(type(terminal.get('delta_count')) is int and terminal['delta_count'] == len(parts), 'count mismatch')
    require(type(terminal.get('text_bytes')) is int and terminal['text_bytes'] == len(text.encode('utf-8')), 'bytes mismatch')
    require(terminal.get('sha256') == text_hash(text), 'hash mismatch')
    return {'transport_complete':True,'generation_state':terminal['generation_state'],
            'task_success':None}

print('参考C通过：', check_exercise_c(verify_done_ref))

参考C通过： 7


## 15．在两个终端使用本课服务

终端 A：

```bash
python lesson_49_streaming_sse_and_backpressure.py --serve --port 8766
```

终端 B：设置终端 A 打印的随机凭证。Bash：

```bash
export LESSON49_API_KEY='复制实际api_key'
```

PowerShell：

```powershell
$env:LESSON49_API_KEY = '复制实际api_key'
```

提交并实时显示正文：

```bash
python lesson_49_streaming_sse_and_backpressure.py --client http://127.0.0.1:8766 --prompt "解释流式输出" --key request-001
```

查询已有请求：

```bash
python lesson_49_streaming_sse_and_backpressure.py --client http://127.0.0.1:8766 --request-id 实际ID
```

请求停止：在另一终端使用相同凭证和 `--request-id 实际ID --cancel`。演示输出很短，可能已经结束；终态不会被后来取消改写。

创建新工作用新的幂等键。重复旧键不会重新显示一份完整答案。`--events` 只允许订阅尚未被认领的请求，不能恢复旧流。

程序不把凭证放入 URL 或输出报告；日志与终端仍属于你的本机信任范围。

## 16．可选真实 Qwen 流式推理

固定模型：`Qwen/Qwen3-0.6B`，revision `c1899de289a04d12100db370d81485cdf75e47ca`；使用配套模板、关闭 thinking。默认 CPU 为 FP32、明确 CUDA 为 FP16，不启用量化。

沿用已有可用的 CUDA PyTorch，必要时在独立环境安装本课固定接口版本：

```bash
python -m pip install "transformers==4.57.3" safetensors
python lesson_49_streaming_sse_and_backpressure.py --serve --backend qwen --device cuda --port 8766
```

然后沿用相同的客户端。权重只在服务启动时加载一次。CUDA 不可用或依赖缺失会明确报错，不回退成演示回答。已有完整缓存时可以加 `--local-files-only`。

这是课程接口版本，不是最新版声明，也不是制作环境已验证的兼容矩阵。真实运行需要下载模型；本讲义保存的输出没有真实模型成绩。

下面只在显式开启时执行真实路径。首次下载和加载可能需要较多内存；模型推理无需同时启动额外的服务实例。

In [29]:
RUN_REAL_STREAM = False
REAL_DEVICE = 'cpu'
REAL_LOCAL_FILES_ONLY = False

if RUN_REAL_STREAM:
    real_backend = QwenBackend(REAL_DEVICE, REAL_LOCAL_FILES_ONLY)
    with local_server(real_backend) as (real_service, real_url):
        start = time.monotonic()
        status, obj = json_call(real_url, real_service.api_key, 'POST', '/v1/requests',
            request_body('用两句话解释KV cache。', 96, 120000), 'real-001')
        require(status == 202, 'real submission failed')
        real_result = consume_stream(real_url, real_service.api_key, obj['request_id'],
                                     print_deltas=True, started_at=start)
        print('\n', real_result)
        print(real_service.get(obj['request_id']).snapshot())
else:
    print('真实模型路径未开启，没有执行Qwen或CUDA推理。')

真实模型路径未开启，没有执行Qwen或CUDA推理。


## 17．完成标准与局限

本课应能独立解释：

1. 为什么一次 `read()` 不等于一个事件，为什么“中”可以分三次到达而不乱码？
2. 缓冲满时，为什么不丢一段正文继续声称成功？
3. 为什么取消后执行槽要等后端退出才能释放？
4. 为什么服务端所有写调用返回，仍不能证明客户端收全？
5. 为什么 `Last-Event-ID` 不是自动断点续传？

真实本机测试已经覆盖协议与控制逻辑，但没有浏览器、Nginx、公网、Windows、真实模型或 GPU 性能验证。`http.server` 官方不建议用于生产；本课也没有 TLS、多租户、总连接数限流、持久化、硬取消、断线重放和连续批处理。[http.server 边界](https://docs.python.org/3.13/library/http.server.html)

流式文字可能还是未完成的 JSON、引用或代码：**只展示为部分草稿，不在收到最终合法结果并完成原有校验前执行工具。** HTTP 200、收到 done、任务正确，是不同结论。

### 下一课
第 50 课进入**项目集成与端到端回归**：把检索、模型、工具、审批与服务日志连接成一个可检查的小型助手，并用已有评测判定改动的收益与退步。

## 官方资料与实现参照

- [WHATWG：SSE格式、事件派发和重连](https://html.spec.whatwg.org/multipage/server-sent-events.html)
- [Python：增量编码与解码](https://docs.python.org/3.13/library/codecs.html)
- [Python：同步队列与容量](https://docs.python.org/3.13/library/queue.html)
- [Python：socket超时、发送与关闭](https://docs.python.org/3.13/library/socket.html)
- [Python：http.client](https://docs.python.org/3.13/library/http.client.html)
- [Python：http.server与生产使用警告](https://docs.python.org/3.13/library/http.server.html)
- [Transformers 4.57.3：streamers](https://huggingface.co/docs/transformers/v4.57.3/en/internal/generation_utils)
- [固定版本的TextStreamer源码](https://github.com/huggingface/transformers/blob/v4.57.3/src/transformers/generation/streamers.py)
- [固定版本Qwen3模型卡](https://huggingface.co/Qwen/Qwen3-0.6B/blob/c1899de289a04d12100db370d81485cdf75e47ca/README.md)

报告内也保存 `SOURCES`，便于以后对照接口变更。